## **Importing Libraries**

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df=spark.table("workspace.bronze.dwh_prd_info")
df.display()

## **Trimming String columns**

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType,StringType):
        df=df.withColumn(field.name,trim(field.name))
df.display()

## **Handling nulls of prd_cost**

In [0]:
df=df.fillna(0,subset=['prd_cost'])

## **Standardize prd_line**

In [0]:
df=df.withColumn('prd_line',
              when(upper(col('prd_line'))=='M',"Mountain")
              .when(upper(col('prd_line'))=='R',"Road")
              .when(upper(col('prd_line'))=='S',"Sales")
              .when(upper(col('prd_line'))=='T',"Touring")
              .otherwise('N/A')
              )

df.display()





In [0]:
df.filter(col("prd_end_dt").isNull()).display()

## **Casting datetype**

In [0]:
df=df.withColumn("prd_end_dt",col("prd_end_dt").cast("date")).withColumn("prd_start_dt",col("prd_start_dt").cast("date"))


In [0]:
df.display()

## **Creating new field using prd_key**

In [0]:
df=df.withColumn("cat_id", regexp_replace(substring(col("prd_key"), 1, 5), "-", "_"))\
    .withColumn(
     'prd_key',substring(col('prd_key'),7,length(col("prd_key")))
     )



## **Renaming Column Names**

In [0]:
RENAME_MAP = {
    "prd_id": "product_id",
    "cat_id": "category_id",
    "prd_key": "product_number",
    "prd_nm": "product_name",
    "prd_cost": "product_cost",
    "prd_line": "product_line",
    "prd_start_dt": "start_date",
    "prd_end_dt": "end_date"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

In [0]:
df.display()

## **Writing to silver schema**

In [0]:
df.write.mode('overwrite').format('delta').saveAsTable('workspace.silver.crm_product')

In [0]:
%sql
select * from workspace.silver.crm_product